# 09 - Image Feature Extraction from Histology

## Learning objectives
1. Extract a local H&E patch centered on each Visium spot, sized from the spot diameter.
2. Compute handcrafted features: RGB mean/std, H&E (stain) proxies, GLCM texture, local
   entropy, edge density, tissue fraction.
3. Store features per spot and visualize them spatially.
4. See `squidpy`'s built-in extractor as an alternative.

## Concept
This is **radiomics at microscopy scale**. For each spot we crop the image patch that the
spot physically covered and summarize its morphology with classic features - the same
intensity/texture/edge descriptors you would compute from a CT ROI. These become a
feature vector per spot, perfectly aligned with the expression matrix (same rows).


In [ ]:
# --- Standard setup: make `utils` importable and seed RNGs ---
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / 'utils').exists():
    ROOT = ROOT.parent  # in case the notebook is opened from a subfolder
sys.path.insert(0, str(ROOT))

from utils import st_helpers as st
st.set_seeds()  # reproducibility (seed = 0)
print('Project root:', st.project_root())


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

adata = st.load_adata('adata_clustered.h5ad')
sf = st.get_scalefactors(adata)
hires = st.get_image(adata, 'hires')

# uint8 RGB image and hires-space spot coordinates + patch size.
img = np.asarray(hires)
if img.dtype != np.uint8:
    img = (255 * np.clip(img, 0, 1)).astype(np.uint8)
img = img[..., :3]
scalef = sf['tissue_hires_scalef']
coords_hires = adata.obsm['spatial'] * scalef
patch = int(round(sf['spot_diameter_fullres'] * scalef))  # ~spot size in hires px
patch = max(patch, 8)
half = patch // 2
print('Image:', img.shape, '| patch size (hires px):', patch)


### Per-spot patch feature function
We pull a square patch around each spot center and compute a fixed feature set. Spots whose
patch would fall off the image are clipped to the border.

In [ ]:
from skimage.color import rgb2hed, rgb2gray
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import sobel
from skimage.filters.rank import entropy as rank_entropy
from skimage.morphology import disk
from skimage.util import img_as_ubyte

def patch_features(p):
    """Compute handcrafted features for one RGB uint8 patch."""
    feats = {}
    # --- color intensity (mean/std per channel) ---
    for i, c in enumerate('rgb'):
        feats[f'mean_{c}'] = float(p[..., i].mean())
        feats[f'std_{c}'] = float(p[..., i].std())
    # --- H&E stain proxies (nuclei vs cytoplasm density) ---
    hed = rgb2hed(p / 255.0)
    feats['hematoxylin_mean'] = float(hed[..., 0].mean())  # nuclei
    feats['eosin_mean'] = float(hed[..., 1].mean())        # cytoplasm/ECM
    # --- grayscale + texture (GLCM Haralick) ---
    gray = img_as_ubyte(rgb2gray(p))
    glcm = graycomatrix(gray, distances=[1], angles=[0, np.pi/2],
                        levels=256, symmetric=True, normed=True)
    for prop in ['contrast', 'homogeneity', 'energy', 'correlation']:
        feats[f'glcm_{prop}'] = float(graycoprops(glcm, prop).mean())
    # --- local entropy (texture randomness) ---
    feats['entropy_mean'] = float(rank_entropy(gray, disk(3)).mean())
    # --- edge density (Sobel magnitude) ---
    feats['edge_density'] = float(sobel(rgb2gray(p)).mean())
    # --- tissue fraction (non-background pixels in the patch) ---
    feats['tissue_fraction'] = float((gray < 220).mean())
    return feats


In [ ]:
from tqdm import tqdm

H, W = img.shape[:2]
rows = []
for (x, y) in tqdm(coords_hires, desc='spots'):
    xi, yi = int(round(x)), int(round(y))
    x0, x1 = max(0, xi - half), min(W, xi + half)
    y0, y1 = max(0, yi - half), min(H, yi + half)
    p = img[y0:y1, x0:x1]
    if p.shape[0] < 4 or p.shape[1] < 4:  # degenerate border patch
        rows.append({})
        continue
    rows.append(patch_features(p))

feat_df = pd.DataFrame(rows, index=adata.obs_names)
feat_df = feat_df.fillna(feat_df.mean(numeric_only=True))  # fill rare border gaps
print('feature matrix:', feat_df.shape)
feat_df.head()


**Expected output:** a spots x ~15 feature DataFrame (mean/std RGB, hematoxylin/eosin
means, four GLCM texture stats, entropy, edge density, tissue fraction).

### Store features in AnnData and save CSV
We keep the feature matrix in `adata.obsm['img_features']` (aligned to spots) and also copy
a few headline features into `adata.obs` for easy plotting.

In [ ]:
adata.obsm['img_features'] = feat_df.values
adata.uns['img_feature_names'] = list(feat_df.columns)
for col in ['hematoxylin_mean', 'eosin_mean', 'glcm_contrast',
            'entropy_mean', 'edge_density', 'tissue_fraction']:
    adata.obs[f'img_{col}'] = feat_df[col].values

feat_csv = st.outputs_dir() / 'image_features.csv'
feat_df.to_csv(feat_csv)
print('Wrote', feat_csv)


### Visualize image features over the tissue

In [ ]:
import squidpy as sq
sq.pl.spatial_scatter(
    adata,
    color=['img_hematoxylin_mean', 'img_entropy_mean',
           'img_edge_density', 'img_tissue_fraction'],
    ncols=2, size=1.3, cmap='cividis',
)


**Expected output:** feature maps that themselves show tissue structure - e.g. hematoxylin
(nuclear density) and texture features tracing anatomy, echoing the expression maps.

### Alternative: squidpy's `ImageContainer`
Squidpy can extract summary/texture/histogram features for you. It is convenient but the
handcrafted route above is transparent and gives us exactly the radiomics-style features
we want. The squidpy version (not run here to keep dependencies light) looks like:

```python
import squidpy as sq
lib = st.get_library_id(adata)
container = sq.im.ImageContainer(
    st.get_image(adata, 'hires'),
    scale=adata.uns['spatial'][lib]['scalefactors']['tissue_hires_scalef'],
)
sq.im.calculate_image_features(
    adata, container, features=['summary', 'texture', 'histogram'],
    key_added='squidpy_img_features',
)
```

### Note on CNN features
In a research setting you would often replace (or augment) handcrafted features with a CNN
embedding (e.g. a pathology foundation model) of each patch. The pipeline is identical -
swap `patch_features` for `model(patch)` - which is why we kept the patch-extraction loop
modular.


In [ ]:
saved = st.save_adata(adata, 'adata_features.h5ad')
print('Wrote', saved)


## Common pitfalls
- Forgetting the scale factor when converting spot coords to hires pixels (patches land in
  the wrong place) - reuse `coords * tissue_hires_scalef`.
- Patch size from full-res pixels applied to the hires image - must scale it too.
- GLCM on an RGB or float image - convert to a uint8 grayscale first.

## Interpretation
Every spot now has both a gene-expression vector and an image-feature vector, perfectly
aligned. We are ready to relate morphology to molecules.

## What this means biologically
Handcrafted features quantify what a pathologist eyeballs: nuclear density, texture,
edges, cellularity. If these morphological summaries track expression, then routine H&E
carries recoverable molecular information.

---
**Next:** `10_integrating_histology_features_with_gene_expression.ipynb`.
